# Myanmar Sign Language (MSL) Recognition for Emergency Domain
## ONNX Inference - Version 1.0
## Runtime Environment: Local Machine (NVIDIA GeForce GTX 1060)

In [1]:
import numpy as np
from pathlib import Path
from IPython.display import Image, display
from IPython.display import Video

In [2]:
ROOT = Path("..").resolve()

CONFIG = ROOT / "config" / "config.yaml"

DATA_DIR = ROOT / "data"
INFER_VIDS = DATA_DIR / "sample4infer"

SCRIPTS_DIR = ROOT / "scripts"
SRC_DIR = ROOT / "src"

RESULTS_DIR = ROOT / "results"
ONNX_DIR = ROOT / "onnx_models"

## Export .pth Models to ONNX Models

In [3]:
%cd {ROOT}

/home/lawun330/Desktop/myanmar-sign-language-recognition


In [4]:
!python {SRC_DIR}/export_to_onnx.py --all --output_dir {ONNX_DIR}

Device: cuda

  Exporting  BILSTM
  Classes: 541   MaxSeqLen: 200
  BiLSTM dummy inputs: kp=(2, 200, 225), mask=(2, 200)
/home/lawun330/Desktop/myanmar-sign-language-recognition/src/export_to_onnx.py:123: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0624 04:02:32.100000 121606 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 16 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
/home/lawun330/anaconda3/envs/lu_msl_reco

In [5]:
!ls {ONNX_DIR}

bilstm_msl.json       stgcn_msl.json	    transformer_msl.onnx
bilstm_msl.onnx       stgcn_msl.onnx	    transformer_msl.onnx.data
bilstm_msl.onnx.data  stgcn_msl.onnx.data
label_map.json	      transformer_msl.json


## ONNX Inference with Multiple Videos

Sample videos:
- `data/sample4infer/idx20-101.mp4` $\rightarrow$ "ဖုန်းဆက် 199"
- `data/sample4infer/idx20-201.mp4` $\rightarrow$ "ကိုက် မြွေ အဆိပ်ရှိ ဆိုး"
- `data/sample4infer/idx20-301.mp4` $\rightarrow$ "ဖန်ကွဲစ ရှမိပြီး သွေးတွေတအားထွက်နေ"
- `data/sample4infer/idx20-401.mp4` $\rightarrow$ "ကျွန်တော် တစ်ကိုယ်လုံး ကိုက်ခဲနေတယ်"
- `data/sample4infer/idx20-501.mp4` $\rightarrow$ "ပလာစတာ တစ်ခု လိုချင်လို့ပါ"

In [6]:
%cd {INFER_VIDS}

/home/lawun330/Desktop/myanmar-sign-language-recognition/data/sample4infer


In [7]:
!ls

idx20-101.mp4  idx20-201.mp4  idx20-301.mp4  idx20-401.mp4  Idx20-501.mp4


In [8]:
VIDEOS = [f for f in INFER_VIDS.glob("*.mp4")]
VIDEOS

[PosixPath('/home/lawun330/Desktop/myanmar-sign-language-recognition/data/sample4infer/Idx20-501.mp4'),
 PosixPath('/home/lawun330/Desktop/myanmar-sign-language-recognition/data/sample4infer/idx20-401.mp4'),
 PosixPath('/home/lawun330/Desktop/myanmar-sign-language-recognition/data/sample4infer/idx20-301.mp4'),
 PosixPath('/home/lawun330/Desktop/myanmar-sign-language-recognition/data/sample4infer/idx20-201.mp4'),
 PosixPath('/home/lawun330/Desktop/myanmar-sign-language-recognition/data/sample4infer/idx20-101.mp4')]

### BiLSTM

In [9]:
(RESULTS_DIR / "exp_onnx_bilstm").mkdir(parents=True, exist_ok=True)

In [10]:
for video in VIDEOS:
    !python {SRC_DIR}/infer_onnx.py \
        --onnx_model {ONNX_DIR}/bilstm_msl.onnx \
        --video {video} \
        --top_k 5 \
        --output_dir {RESULTS_DIR}/"exp_onnx_bilstm" \
        --log_file {RESULTS_DIR}/"exp_onnx_bilstm"/video_infer_log.txt

[INFO] Loaded 541 labels from label map.
[INFO] ONNX loaded: bilstm_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Logging top-5 results to: /home/lawun330/Desktop/myanmar-sign-language-recognition/results/exp_onnx_bilstm/video_infer_log.txt
[INFO] Saving output videos to: /home/lawun330/Desktop/myanmar-sign-language-recognition/results/exp_onnx_bilstm/
I0000 00:00:1782252530.153598  127385 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1782252530.156330  127433 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics 630 (CFL GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
I0000 00:00:1782252530.164982  127385 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1782252530.167620  127448 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics 630 (CFL GT2)
W0000 00:00:1782252530.2130

### Transformer

In [11]:
(RESULTS_DIR / "exp_onnx_transformer").mkdir(parents=True, exist_ok=True)

In [12]:
for video in VIDEOS:
    !python {SRC_DIR}/infer_onnx.py \
        --onnx_model {ONNX_DIR}/transformer_msl.onnx \
        --video {video} \
        --top_k 5 \
        --output_dir {RESULTS_DIR}/"exp_onnx_transformer" \
        --log_file {RESULTS_DIR}/"exp_onnx_transformer"/video_infer_log.txt


[INFO] Loaded 541 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Logging top-5 results to: /home/lawun330/Desktop/myanmar-sign-language-recognition/results/exp_onnx_transformer/video_infer_log.txt
[INFO] Saving output videos to: /home/lawun330/Desktop/myanmar-sign-language-recognition/results/exp_onnx_transformer/
I0000 00:00:1782253334.677151  128520 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1782253334.680513  128568 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics 630 (CFL GT2)
I0000 00:00:1782253334.687243  128520 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
I0000 00:00:1782253334.690801  128583 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics 630 (CFL GT2)
W0000 00:00:

### ST-GCN

In [13]:
(RESULTS_DIR / "exp_onnx_stgcn").mkdir(parents=True, exist_ok=True)

In [14]:
for video in VIDEOS:
    !python {SRC_DIR}/infer_onnx.py \
        --onnx_model {ONNX_DIR}/stgcn_msl.onnx \
        --video {video} \
        --top_k 5 \
        --output_dir {RESULTS_DIR}/"exp_onnx_stgcn" \
        --log_file {RESULTS_DIR}/"exp_onnx_stgcn"/video_infer_log.txt


[INFO] Loaded 541 labels from label map.
[INFO] ONNX loaded: stgcn_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Logging top-5 results to: /home/lawun330/Desktop/myanmar-sign-language-recognition/results/exp_onnx_stgcn/video_infer_log.txt
[INFO] Saving output videos to: /home/lawun330/Desktop/myanmar-sign-language-recognition/results/exp_onnx_stgcn/
I0000 00:00:1782254939.226717  130086 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1782254939.230049  130135 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics 630 (CFL GT2)
I0000 00:00:1782254939.236226  130086 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
I0000 00:00:1782254939.238128  130150 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics 630 (CFL GT2)
W0000 00:00:1782254939.279521 

## ONNX Inference with Webcam

In [15]:
(RESULTS_DIR / "exp_onnx_transformer" / "webcam").mkdir(parents=True, exist_ok=True)

In [16]:
!python {SRC_DIR}/infer_onnx.py \
    --onnx_model {ONNX_DIR}/transformer_msl.onnx \
    --webcam 0 \
    --no_mirror \
    --buf_frames 100 \
    --zero_z \
    --top_k 5 \
    --output_dir {RESULTS_DIR}/exp_onnx_transformer/webcam \
    --log_file {RESULTS_DIR}/exp_onnx_transformer/webcam/webcam_log.txt

[INFO] Loaded 541 labels from label map.
[INFO] ONNX loaded: transformer_msl.onnx  (provider: CPUExecutionProvider)
[INFO] Logging top-5 results to: /home/lawun330/Desktop/myanmar-sign-language-recognition/results/exp_onnx_transformer/webcam/webcam_log.txt
[INFO] Saving output videos to: /home/lawun330/Desktop/myanmar-sign-language-recognition/results/exp_onnx_transformer/webcam/
I0000 00:00:1782256549.120693  133794 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1782256549.125064  133842 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics 630 (CFL GT2)
I0000 00:00:1782256549.133979  133794 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1782256549.139270  133857 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics 630 (CFL GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W00